# Outliers - Final Adaptive Rocchio Pipeline
The fixed procedure is:

1. Build one TF-IDF index and one sparse TF-IDF matrix.
2. Run two fixed RM3 configurations.
3. Apply fixed pseudo-Rocchio feedback using the top 5 documents and `beta=20`.
4. For queries with at most 8 processed words, use the robust branch.
5. For longer queries, use the best-MAP branch.
6. Save at most 1,000 results per query in TREC format.

All retrieval remains sparse and word based. No qrel information is used to change an individual query.

In [1]:
# Run only in a fresh environment:
# %pip install -q python-terrier pandas numpy scipy

from collections import Counter
from pathlib import Path
import math
import os
import re
import time

os.environ.setdefault("PYTERRIER_HOME", str((Path.cwd() / ".pyterrier").resolve()))

import numpy as np
import pandas as pd
from scipy import sparse
import pyterrier as pt

if not pt.java.started():
    pt.java.init()

print("PyTerrier:", pt.__version__)
print("Java started:", pt.java.started())

PyTerrier: 0.13.1
Java started: True


Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


## 2. Read the Cranfield files

The qrels identify the 225 query blocks sequentially, so the query IDs are assigned from 1 through 225 in file order.

In [2]:
ROOT = Path.cwd()


def find_data_dir():
    candidates = [ROOT, Path("/content"), ROOT / "cranfield"]
    required = {"cran.all.1400", "cran.qry", "cranqrel"}
    for folder in candidates:
        if folder.exists() and required.issubset({path.name for path in folder.iterdir()}):
            return folder.resolve()
    raise FileNotFoundError("Place cran.all.1400, cran.qry, and cranqrel beside this notebook.")


def parse_documents(path):
    records = []
    current_id = None
    section = None
    fields = {"title": [], "abstract": []}

    for raw in path.read_text(encoding="ascii", errors="ignore").splitlines():
        line = raw.strip()
        match = re.fullmatch(r"\.I\s+(\d+)", line)
        if match:
            if current_id is not None:
                records.append({
                    "docno": str(current_id),
                    "title": " ".join(fields["title"]),
                    "abstract": " ".join(fields["abstract"]),
                })
            current_id = int(match.group(1))
            section = None
            fields = {"title": [], "abstract": []}
        elif line == ".T":
            section = "title"
        elif line == ".W":
            section = "abstract"
        elif line in {".A", ".B"}:
            section = None
        elif section in fields:
            fields[section].append(line)

    if current_id is not None:
        records.append({
            "docno": str(current_id),
            "title": " ".join(fields["title"]),
            "abstract": " ".join(fields["abstract"]),
        })

    for record in records:
        record["text"] = record["title"] + " " + record["abstract"]
    return pd.DataFrame(records)


def parse_queries(path):
    texts = []
    words = []
    active = False

    for raw in path.read_text(encoding="ascii", errors="ignore").splitlines():
        line = raw.strip()
        if re.fullmatch(r"\.I\s+\d+", line):
            if words:
                texts.append(" ".join(words))
            words = []
            active = False
        elif line == ".W":
            active = True
        elif active:
            words.append(line)

    if words:
        texts.append(" ".join(words))

    texts = [re.sub(r"[^A-Za-z0-9]+", " ", text).strip().lower() for text in texts]
    return pd.DataFrame({
        "qid": [str(i) for i in range(1, len(texts) + 1)],
        "query": texts,
    })


DATA_DIR = find_data_dir()
docs = parse_documents(DATA_DIR / "cran.all.1400")
queries = parse_queries(DATA_DIR / "cran.qry")
qrels = pd.read_csv(
    DATA_DIR / "cranqrel",
    sep=r"\s+",
    header=None,
    names=["qid", "docno", "label"],
    dtype={"qid": str, "docno": str, "label": int},
)

assert len(docs) == 1400
assert docs["docno"].nunique() == 1400
assert len(queries) == 225
assert qrels["qid"].nunique() == 225
assert set(queries["qid"]) == set(qrels["qid"])
assert qrels["docno"].isin(docs["docno"]).all()

print("Documents:", len(docs))
print("Queries:", len(queries))
print("All data checks passed.")

Documents: 1400
Queries: 225
All data checks passed.


## 3. Build the sparse TF-IDF vectors

These vectors are used by Rocchio. Common words are removed, related word forms are stemmed, and every vector is length-normalized.

In [3]:
STOPWORDS = set("""
        a an and are as at be been being by can could did do does for from had has have how i if in into
        is it its may might must no not of on or our should so such than that the their them then there
        these they this those to under was we were what when where which who why will with would you your
    """.split())
WORD_RE = re.compile(r"[a-z0-9]+")

from jnius import autoclass
porter = autoclass("org.terrier.terms.PorterStemmer")()

unique_words = set()
for text in list(docs["text"]) + list(queries["query"]):
    unique_words.update(WORD_RE.findall(text.lower()))
stem_cache = {word: porter.stem(word) for word in unique_words}


def tokenize(text):
    return [
        stem_cache[word]
        for word in WORD_RE.findall(text.lower())
        if word not in STOPWORDS and len(word) > 1
    ]


def build_document_matrix(texts, min_df=2):
    token_lists = [tokenize(text) for text in texts]
    document_frequency = Counter()
    for row in token_lists:
        document_frequency.update(set(row))

    kept_terms = sorted(
        term for term, frequency in document_frequency.items()
        if frequency >= min_df and frequency <= 0.95 * len(texts)
    )
    vocabulary = {term: index for index, term in enumerate(kept_terms)}

    rows, columns, values = [], [], []
    for row_id, row in enumerate(token_lists):
        for term, count in Counter(term for term in row if term in vocabulary).items():
            rows.append(row_id)
            columns.append(vocabulary[term])
            values.append(count)

    counts = sparse.csr_matrix(
        (values, (rows, columns)),
        shape=(len(texts), len(vocabulary)),
        dtype=float,
    )
    idf = np.array([
        math.log((len(texts) + 1) / (document_frequency[term] + 1)) + 1
        for term in kept_terms
    ])
    counts.data = 1 + np.log(counts.data)
    matrix = counts.multiply(idf).tocsr()
    norms = np.sqrt(matrix.multiply(matrix).sum(axis=1)).A1
    matrix = sparse.diags(1 / np.maximum(norms, 1e-12)) @ matrix
    return matrix, vocabulary, idf


def build_query_matrix(texts, vocabulary, idf):
    rows, columns, values = [], [], []
    for row_id, text in enumerate(texts):
        for term, count in Counter(term for term in tokenize(text) if term in vocabulary).items():
            rows.append(row_id)
            columns.append(vocabulary[term])
            values.append(count)

    matrix = sparse.csr_matrix(
        (values, (rows, columns)),
        shape=(len(texts), len(vocabulary)),
        dtype=float,
    )
    matrix.data = 1 + np.log(matrix.data)
    matrix = matrix.multiply(idf).tocsr()
    norms = np.sqrt(matrix.multiply(matrix).sum(axis=1)).A1
    return sparse.diags(1 / np.maximum(norms, 1e-12)) @ matrix


vector_start = time.perf_counter()
document_matrix, vocabulary, idf = build_document_matrix(docs["text"])
query_matrix = build_query_matrix(queries["query"], vocabulary, idf)
vector_time = time.perf_counter() - vector_start

docnos = docs["docno"].to_numpy()
doc_to_index = {docno: index for index, docno in enumerate(docnos)}

print("Sparse matrix shape:", document_matrix.shape)
print(f"Vector construction time: {vector_time:.3f} seconds")

Sparse matrix shape: (1400, 2919)
Vector construction time: 0.075 seconds


## 4. Build the Terrier index and run the two fixed RM3 branches

- **Best-MAP branch:** 30 feedback documents, 20 terms, lambda 0.6.
- **Robust branch:** 30 feedback documents, 35 terms, lambda 0.4.

In [4]:
work_dir = ROOT / ".outliers_final_work"
work_dir.mkdir(exist_ok=True)

index_start = time.perf_counter()
indexer = pt.IterDictIndexer(
    str(work_dir / "tfidf_index"),
    meta={"docno": 10},
    overwrite=True,
)
index = pt.IndexFactory.of(indexer.index(
    {"docno": row.docno, "text": row.text}
    for row in docs.itertuples()
))
index_time = time.perf_counter() - index_start

tfidf = pt.terrier.Retriever(
    index,
    wmodel="TF_IDF",
    controls={"c": "0.75"},
    num_results=1000,
)


def run_rm3(fb_docs, fb_terms, fb_lambda):
    pipeline = (
        tfidf
        >> pt.rewrite.RM3(
            index,
            fb_docs=fb_docs,
            fb_terms=fb_terms,
            fb_lambda=fb_lambda,
        )
        >> tfidf
    )
    return pipeline.transform(queries)


search_start = time.perf_counter()
best_rm3_run = run_rm3(fb_docs=30, fb_terms=20, fb_lambda=0.6)
robust_rm3_run = run_rm3(fb_docs=30, fb_terms=35, fb_lambda=0.4)

print(f"Indexing time: {index_time:.3f} seconds")

17:55:51.754 [ForkJoinPool-1-worker-3] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (471) - further warnings are suppressed
17:55:52.071 [ForkJoinPool-1-worker-3] WARN org.terrier.structures.indexing.Indexer -- Indexed 2 empty documents
Indexing time: 0.718 seconds


## 5. Apply the fixed Rocchio step

For each query, the normalized TF-IDF vectors of the first five RM3 documents are averaged. The query vector is combined with that average using `beta=20`, then all documents are ranked by cosine similarity.

In [5]:
def scores_to_run(scores, depth=1000):
    parts = []
    for query_index, qid in enumerate(queries["qid"]):
        order = np.argsort(-scores[query_index], kind="stable")[:depth]
        order = order[scores[query_index, order] > 0]
        parts.append(pd.DataFrame({
            "qid": qid,
            "docno": docnos[order],
            "score": scores[query_index, order],
            "rank": np.arange(len(order)),
        }))
    return pd.concat(parts, ignore_index=True)


def fixed_rocchio(initial_run, topk=5, beta=20):
    grouped = {
        qid: group.sort_values("rank")
        for qid, group in initial_run.groupby("qid")
    }
    scores = np.zeros((len(queries), len(docs)))

    for query_index, qid in enumerate(queries["qid"]):
        feedback_docnos = grouped[qid].head(topk)["docno"]
        feedback_indices = [doc_to_index[docno] for docno in feedback_docnos]

        feedback_centroid = np.asarray(
            document_matrix[feedback_indices].mean(axis=0)
        ).ravel()
        updated_query = query_matrix[query_index].toarray().ravel()
        updated_query = updated_query + beta * feedback_centroid

        norm = np.linalg.norm(updated_query)
        if norm > 0:
            updated_query /= norm
        scores[query_index] = document_matrix.dot(updated_query)

    return scores_to_run(scores, depth=1000)


best_rocchio_run = fixed_rocchio(best_rm3_run)
robust_rocchio_run = fixed_rocchio(robust_rm3_run)

# Short queries use the robust branch; longer queries use the best-MAP branch.
query_lengths = queries.set_index("qid")["query"].map(lambda text: len(tokenize(text)))
short_qids = set(query_lengths[query_lengths <= 8].index)

final_run = pd.concat([
    robust_rocchio_run[robust_rocchio_run["qid"].isin(short_qids)],
    best_rocchio_run[~best_rocchio_run["qid"].isin(short_qids)],
])
final_run = final_run.sort_values(["qid", "rank"]).reset_index(drop=True)
search_time = time.perf_counter() - search_start

print("Short queries using robust branch:", len(short_qids))
print("Long queries using best-MAP branch:", len(queries) - len(short_qids))
print(f"Total search and feedback time: {search_time:.3f} seconds")

Short queries using robust branch: 74
Long queries using best-MAP branch: 151
Total search and feedback time: 10.890 seconds


## 6. Evaluate once and export the final result file

In [6]:
METRICS = ["map", "recip_rank", "P.5", "P.10", "ndcg_cut.10"]


def evaluate(run):
    return pt.Evaluate(run, qrels, metrics=METRICS)


summary = pd.DataFrame([
    {"Method": "RM3 baseline", **evaluate(best_rm3_run)},
    {"Method": "Final adaptive Rocchio", **evaluate(final_run)},
])
display(summary)

output_path = ROOT / "Outliers_Final_Adaptive_Rocchio_results.txt"
submission = (
    final_run.sort_values(["qid", "rank"])
    .groupby("qid", group_keys=False)
    .head(1000)
    .copy()
)
submission["Q0"] = "Q0"
submission["rank"] = submission["rank"] + 1
submission["tag"] = "Outliers_Final_Adaptive_Rocchio"
submission[["qid", "Q0", "docno", "rank", "score", "tag"]].to_csv(
    output_path,
    sep=" ",
    header=False,
    index=False,
)

assert submission["qid"].nunique() == 225
assert not submission.duplicated(["qid", "docno"]).any()
assert submission.groupby("qid").size().max() <= 1000

print("Saved:", output_path)
print("Rows:", len(submission))
print("Queries:", submission["qid"].nunique())
print("Maximum results per query:", submission.groupby("qid").size().max())

,Method,map,recip_rank,P.5,P.10,ndcg_cut.10
0,RM3 baseline,0.347084,0.551114,0.360889,0.271111,0.377596
1,Final adaptive Rocchio,0.388018,0.591465,0.361778,0.286222,0.409090


Saved: /home/tharun/Documents/IR/A2/Outliers_Final_Adaptive_Rocchio_results.txt
Rows: 225000
Queries: 225
Maximum results per query: 1000
